# Mini-Evaluation (S1 vs S2 vs S3 vs S4)
This notebook runs a small ablation evaluation with 2 queries per FA-Type.

In [1]:
import os
import sys
import pandas as pd
from pathlib import Path

# Resolve project root regardless of Jupyter's starting CWD
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Could not locate project root (pyproject.toml)")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.common.utils import setup_logging
from src.common.ingestion import download_all_filings
from src.systems.rag_monolith.pipeline import MonolithRAGPipeline
from src.systems.rag_agent.pipeline import AgentRAGPipeline
from src.systems.long_context.pipeline import LongContextPipeline
from src.systems.multi_agent.pipeline import MultiAgentPipeline

setup_logging()
GOLD_CSV = PROJECT_ROOT / "data" / "gold_standard" / "gold_standard_v3.csv"

Project root: /Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis


In [2]:
# Load 2 queries of each type
df_gs = pd.read_csv(GOLD_CSV, sep=";")
df_sample = (
    df_gs.groupby("fa_type", group_keys=False)
    .sample(n=2, random_state=42)
    .reset_index(drop=True)
)
queries = df_sample["query"].tolist()
display(df_sample[["id", "fa_type", "query"]])
print(f"Loaded {len(df_sample)} queries from {GOLD_CSV.relative_to(PROJECT_ROOT)}")


,id,fa_type,query
0,85,FA-1,Wie hoch war das Operating Income von GOOGL in...
1,16,FA-1,Wie hoch war der Total Revenue von GOOGL in FY...
2,21,FA-2,Wie hoch war die Operating Margin von Apple in...
3,26,FA-2,Wie hoch war die Operating Margin von MSFT in ...
4,45,FA-3,Wie haben sich die R&D-Ausgaben von Apple von ...
5,106,FA-3,Um wie viele Prozentpunkte hat sich die Gross ...
6,61,FA-4,Welches Unternehmen hatte mehr Net Income in F...
7,59,FA-4,Welches Unternehmen hatte das höhere Net Incom...
8,147,FA-Refusal,Wie hoch war der Cashflow?
9,124,FA-Refusal,Wie hoch war das Net Income von MSFT in FY2020?


Loaded 10 queries from data/gold_standard/gold_standard_v3.csv


In [3]:
# Load SEC filings (downloads/parses if not cached)
filings = download_all_filings()
print(f"Loaded {len(filings)} filings.")

2026-07-25 14:41:36 [INFO] src.common.ingestion: ==================================================
2026-07-25 14:41:36 [INFO] src.common.ingestion: Processing Apple Inc. (AAPL)
2026-07-25 14:41:36 [INFO] src.common.ingestion: ==================================================
2026-07-25 14:41:36 [INFO] src.common.ingestion: ⏭ Skipping download — AAPL FYlatest already exists: 10K_2024-11-01.md
2026-07-25 14:41:36 [INFO] src.common.ingestion: ✓ AAPL: 5 sections extracted, 200703 chars total
2026-07-25 14:41:36 [INFO] src.common.ingestion: ==================================================
2026-07-25 14:41:36 [INFO] src.common.ingestion: Processing Microsoft Corporation (MSFT)
2026-07-25 14:41:36 [INFO] src.common.ingestion: ==================================================
2026-07-25 14:41:36 [INFO] src.common.ingestion: ⏭ Skipping download — MSFT FYlatest already exists: 10K_2024-07-30.md
2026-07-25 14:41:36 [INFO] src.common.ingestion: ✓ MSFT: 5 sections extracted, 335856 chars total

Loaded 4 filings.


In [4]:
# Initialize and build pipelines
print("Building S1 (RAG Monolith)...")
s1 = MonolithRAGPipeline()
s1.build(filings)

print("Building S2 (Agent RAG)...")
s2 = AgentRAGPipeline()
s2.build(filings)

print("Building S3 (Long Context)...")
s3 = LongContextPipeline()
s3.build(filings)

print("Building S4 (Multi-Agent)...")
s4 = MultiAgentPipeline()
s4.build(filings)


2026-07-25 14:41:36 [INFO] src.systems.rag_monolith.pipeline: Building pipeline: chunk_size=1000, overlap=10%, bm25_weight=0.50, pre_k=15
2026-07-25 14:41:36 [INFO] src.systems.rag_monolith.chunker: Chunked AAPL FY2024: 288 chunks (chunk_size=1000, overlap=100)
2026-07-25 14:41:36 [INFO] src.systems.rag_monolith.chunker: Chunked MSFT FY2024: 734 chunks (chunk_size=1000, overlap=100)
2026-07-25 14:41:36 [INFO] src.systems.rag_monolith.chunker: Chunked AMZN FY2024: 1135 chunks (chunk_size=1000, overlap=100)
2026-07-25 14:41:36 [INFO] src.systems.rag_monolith.chunker: Chunked GOOGL FY2024: 1367 chunks (chunk_size=1000, overlap=100)
2026-07-25 14:41:36 [INFO] src.systems.rag_monolith.chunker: Total: 1367 documents from 4 filings (chunk_size=1000, overlap_pct=10%)
/Users/danielkojok/Documents/Uni/Masterarbeit/agentic-financial-report-analysis/src/common/llm_client.py:119: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0

Building S1 (RAG Monolith)...


2026-07-25 14:41:54 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:42:01 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:42:09 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:42:16 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:42:23 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/

Building S2 (Agent RAG)...


2026-07-25 14:42:48 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:42:55 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:43:03 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:43:09 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:43:18 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/

Building S3 (Long Context)...
Building S4 (Multi-Agent)...


In [5]:
# Run evaluation
results = []

for i, row in df_sample.iterrows():
    q = row['query']
    fa_type = row['fa_type']
    print(f"\n{'='*50}\nQuery [{fa_type}]: {q}")
    
    r1 = s1.query(q)
    print(f"[S1] {r1.metrics.token_usage.total_tokens} tokens | Ans: {r1.answer[:100]}...")
    
    r2 = s2.query(q)
    print(f"[S2] {r2.metrics.token_usage.total_tokens} tokens | Ans: {r2.answer[:100]}...")
    
    r3 = s3.query(q)
    print(f"[S3] {r3.metrics.token_usage.total_tokens} tokens | Ans: {r3.answer[:100]}...")
    
    r4 = s4.query(q)
    print(f"[S4] {r4.metrics.token_usage.total_tokens} tokens | Ans: {r4.answer[:100]}...")
    
    results.append({
        'id': row['id'],
        'fa_type': fa_type,
        'query': q,
        'S1_tokens': r1.metrics.token_usage.total_tokens,
        'S1_ans': r1.answer,
        'S2_tokens': r2.metrics.token_usage.total_tokens,
        'S2_ans': r2.answer,
        'S3_tokens': r3.metrics.token_usage.total_tokens,
        'S3_ans': r3.answer,
        'S4_tokens': r4.metrics.token_usage.total_tokens,
        'S4_ans': r4.answer
    })



Query [FA-1]: Wie hoch war das Operating Income von GOOGL in FY2024?


2026-07-25 14:43:36 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:43:36 [INFO] src.common.retrieval: Hybrid retrieval: 16 documents
2026-07-25 14:43:36 [INFO] src.common.retrieval: FlashRank ranker loaded: ms-marco-MiniLM-L-12-v2
2026-07-25 14:43:36 [INFO] src.common.retrieval: Final retrieval: 4 documents


[S1] 1470 tokens | Ans: Das Operating Income von GOOGL (Alphabet Inc.) betrug im Geschäftsjahr 2024 112.390 Millionen US-Dol...


2026-07-25 14:43:52 [INFO] httpx: HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/project-5d59df8f-e38f-4c57-9a7/locations/global/publishers/google/models/gemini-embedding-001:predict "HTTP/1.1 200 OK"
2026-07-25 14:43:52 [INFO] src.systems.rag_agent.tools.search_section: search_section: GOOGL FY2024 Financial Statements (sub_query='Operating Income') → 4/15 chunks
2026-07-25 14:43:55 [INFO] src.systems.rag_agent.pipeline: Reflection verdict: status=accept, issues=[]
2026-07-25 14:43:55 [INFO] src.systems.rag_agent.pipeline: Agent query completed: 1 tool calls, 15.24s, 8674 tokens (reflection=on, revised=False)


[S2] 8674 tokens | Ans: [{'type': 'text', 'text': 'Das Operating Income (Total income from operations) von GOOGL in FY2024 betrug 112.390 Millionen US-Dollar (GOOGL, 2024, Financial Statements).', 'thought_signature': 'CrYDAY89a1+lZaxmMM6NHML6nS3ZThUHmt4DOoe4V2px3yKtAmiVJOS9EDTDRCyCKiinNswKUwYca7DWT1QuGPShV+KATnrmh0HpkKUBo+qT2kAcMAgU3tVjyZ1hI+hv7aDiCosMLuyEDqfAGB+0kH45JZCDU7KljouYLzHyet+KqfJEtJu2TrRoiA9g8YtpQ+AHCOOs8pUGodLuhtxc35c8SCeQ8Pjv5MQIGehPA6CbLrvniKap/HmiKVzGpHIY4Lawuz0RVyi9YrYu9YtiPk6194JPPGLSDgoGMCRCOCIKRQcXLFM4KoH5mUUyNw2AHMUKeBI5N+ztCRZcIv973pE1soxEGUGBCV5+kwRAJPgUHGdPAPir7ONp7UFSX+UPBaiCrR4C5Z5VccrYp8FSxgYx8kJJYNJUEJLFxEAYJWWSifAEczNQ/2Q65RrgB5xJtyhJO/ye54ZVwwQTDov8l8N1FFUJ9pjDekKqku7MD1Zg1cmFI4WWX/zOfcovMO40kmmV2+r3cQz2X0vhP/7DZPPHocLgGvB7VnkE+qKBo17BWkyNGWvPgpB5vDbiVu8XjfHUhpKEk1Sp'}]...


...

[S2] 7063 tokens | Ans: [{'type': 'text', 'text': 'Es tut mir leid, aber ich habe keine Informationen über das Nettoergebnis von MSFT für das Geschäftsjahr 2020. Die verfügbaren Unterlagen für MSFT umfassen nur das Geschäftsjahr 2024.', 'thought_signature': 'CssCAY89a1+lqVysdmb5FDM1VKt0ScB9YINBV+iM/9u/q/7+DtTunxuhHVqbWWUOvKB3zwRKwLlV931VgQPZDgoDwHF/N5CoQuzOWzSN2SpTmDjnW/SPZfTo0GjzfnB9ErnPu8NCLLv201Tw9966hE1tD59g/vYAZhJ444g0wxNYtOhPLCwO/xW1LRJYdLDz7mZ2RmEVEwxG67G+ApznGZuff9lBonGQCbSEn/LF97lCbqt+HzH5X9rzapiG3xzQa2zJuZjWxV9v5nlIIiMeQu5tUn9aoYhA3NgJeE0QIw8QKSMilEMZvhBf8HNGH/JGP1UEXhgepciSjQAnnzOc0QDQteGTQvBvT43reK8Y+VNnWwS/AWvDomDnSH77y0O7h0ciykf/RRg857hOKxCcTPugRboo2/n5QMCuTYN3aDGONnJX7klHUXVcKKWAdQ=='}]...


2026-07-25 14:57:23 [INFO] src.systems.long_context.pipeline: S3 reflection verdict: status=revise, issues=['unsupported_claim']
2026-07-25 14:57:28 [INFO] src.systems.rag_agent.tools.list_filings: list_filings: returning overview of 4 filings
2026-07-25 14:57:32 [INFO] src.systems.long_context.pipeline: S3 query done: 1 tool calls, 17.92s, 637270 tokens (reflection=on, revised=True)
2026-07-25 14:57:32 [INFO] src.systems.rag_agent.tools.list_filings: list_filings: returning overview of 4 filings
2026-07-25 14:57:32 [INFO] src.systems.long_context.agent: S3 agent built: 2 tools, recursion_limit=12, prompt_chars=2135


[S3] 637270 tokens | Ans: [{'type': 'text', 'text': 'Die verfügbaren Unterlagen enthalten keine Informationen über das Nettoergebnis von MSFT für das Geschäftsjahr 2020. Es sind nur Daten für das Geschäftsjahr 2024 verfügbar (MSFT, FY2024).', 'thought_signature': 'CoMCAY89a19io3Ny6KwOf/sTgiQYlNOlZz4YQbFaN96vXKHcK8glTSGAxaYI2iAxkAQ0Bf2PUz2KovAK8cfWy6r+MUoogUbx3DP+4uTS1vYkkOWvW7zOWNvFnWcXmA/C2k7rbvm0uffDxSHOzLptKl8Ql9b07Y12rK3Rz+66jFBtN35X2A5un9mTJTMfFkdGD3XWVitFYt/Yes5MoVS4bv8lpA0Wn+vEzh8hlImnnyn15QPreKiLLhCnjp661XBJWm/m5PJqAFBl58Sg0OxjgZZKpLfIthHYxrvLeUsWNiPf1/LU88mNuEV21zrbyidJNFiaIYCoMMNU6m7ChfH94qD5WUMeJg=='}]...


2026-07-25 14:57:33 [INFO] src.systems.long_context.agent: S3 agent built: 1 tools, recursion_limit=8, prompt_chars=1025
2026-07-25 14:57:36 [INFO] src.systems.long_context.agent: S3 agent built: 1 tools, recursion_limit=8, prompt_chars=1025
2026-07-25 14:57:39 [INFO] src.systems.long_context.agent: S3 agent built: 1 tools, recursion_limit=8, prompt_chars=1025
2026-07-25 14:57:42 [INFO] src.systems.long_context.agent: S3 agent built: 1 tools, recursion_limit=8, prompt_chars=1025
2026-07-25 14:57:54 [INFO] src.systems.long_context.agent: S3 agent built: 1 tools, recursion_limit=8, prompt_chars=1025
2026-07-25 14:57:57 [INFO] src.systems.long_context.agent: S3 agent built: 1 tools, recursion_limit=8, prompt_chars=1025


[S4] 29506 tokens | Ans: [{'type': 'text', 'text': 'Based on the provided information, no specialist outputs were made available regarding the Net Income of MSFT in FY2020. Therefore, I cannot provide a specific numerical answer to your question. The `calculate` tool cannot be used as there are no numbers provided by specialists to perform calculations on.', 'thought_signature': 'CqJHAY89a1//V1MYFIWg/r9HALI8UXWTrKxZaRDHgaCDXgDQzinIalSuTH9jnvqueim+6oImTtYAr8ud9Wy8x24GvmJ2sLn2DTTqPOkKodFyJuSnQgq5pYpfl2vL3iEy+i4XfPcnwncJZ/aju8tXTXX8qzbNmbny9gCBVfjUltDbM7Z5BH2TQ5cdjS+Uh5hHaAacC7PbSkTvAn2eWd0tezX7IGhzFx4o5DcfgwvEMm6D2Lta4f1t9/VtXBxvzSnqytwDjS+eV9ge/0Jc7wBHHHVw6K2WPvZbIedJLRpicY1+W6LLwXBdpohdhf5qd7SSS8aMDJzYJwAgB/X9Tt200/ZrCOTYE8pyQHs25trxJySLvGo3WjHyFwZLS2swLmrb3me143OUwsgjSc+WIHNVqmoeZVWwiTzHUrqrBx9vgm+08IWXnlZ6a8YBCCHV4cP0YgfyCDkrnWLpRspQv8zcCd2LJ2Jbz1zNy8kvyrbT8LWgG46pDmL+jb4diLt7OGZY0DnAbx83+/3xBBCkz3FO6goDqBmJjAn8d08Qm3terVaMvSosKFvWXQv5xJZ/F5VsfnFuxGgpEiK6J6WgjTctQYg5slsvSeUenEplb8Ayo

In [6]:
# Save and display results
RESULTS_CSV = PROJECT_ROOT / "data" / "results" / "mini_eval_s1_s4_results.csv"
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_CSV, index=False, sep=";")
display(df_results[["fa_type", "S1_tokens", "S2_tokens", "S3_tokens", "S4_tokens"]])
print(f"Saved: {RESULTS_CSV.relative_to(PROJECT_ROOT)}")


,fa_type,S1_tokens,S2_tokens,S3_tokens,S4_tokens
0,FA-1,1470,8674,424674,68003
1,FA-1,1525,7517,424652,78878
2,FA-2,1635,59953,848880,45389
3,FA-2,2533,12048,213670,83962
4,FA-3,1530,10390,849463,30013
5,FA-3,1166,24512,849604,51311
6,FA-4,2194,11646,426439,74665
7,FA-4,1019,13113,638508,64023
8,FA-Refusal,911,2591,212947,29848
9,FA-Refusal,1993,7063,637270,29506


Saved: data/results/mini_eval_s1_s4_results.csv
